# Portfolio model summary

This notebook explains the final merchant portfolio model and provides compact checks of the delivered dashboard outputs. It uses the final CSV files only and does not rebuild or change the model.

The model selects exactly **100 merchants**. Results are available for one overall portfolio and five category portfolios at five minimum expected-revenue retention levels: 100%, 99.5%, 99%, 98%, and 95%.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

# Works when run from either the repository root or the code directory.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
FINAL_DIR = PROJECT_ROOT / "handoff" / "final"

DASHBOARD_FILES = {
    "Overall": "portfolio_dashboard.csv",
    "Arts, Crafts, Gifts & Entertainment": "arts_portfolio_dashboard.csv",
    "Automotive, Outdoor & Recreational Equipment": "auto_portfolio_dashboard.csv",
    "Fashion, Jewellery & Personal Care": "fashion_portfolio_dashboard.csv",
    "Home, Garden & General Supplies": "home_portfolio_dashboard.csv",
    "Technology, Digital & Communications": "tech_portfolio_dashboard.csv",
}

dashboards = {
    label: pd.read_csv(FINAL_DIR / filename, dtype={"merchant_abn": "string"})
    for label, filename in DASHBOARD_FILES.items()
}

pd.DataFrame(
    {
        "dashboard": label,
        "rows": len(frame),
        "portfolios": frame["portfolio_id"].nunique(),
        "merchants_per_portfolio": int(frame.groupby("portfolio_id").size().min()),
    }
    for label, frame in dashboards.items()
)

,dashboard,rows,portfolios,merchants_per_portfolio
0,Overall,500,5,100
1,"Arts, Crafts, Gifts & Entertainment",500,5,100
2,"Automotive, Outdoor & Recreational Equipment",500,5,100
3,"Fashion, Jewellery & Personal Care",500,5,100
4,"Home, Garden & General Supplies",500,5,100
5,"Technology, Digital & Communications",500,5,100


## 1. Expected revenue

The dashboard calls the model's revenue measure **expected monthly gross fees**. For merchant $i$:

$$E_i = \left(\frac{1}{M}\sum_{m=1}^{M} \text{monthly sales}_{i,m}\right) \times \frac{\text{take rate}_i}{100}$$

The delivered snapshot uses 19 complete calendar months from March 2021 through September 2022. Genuine zero-sales months remain in the average. A merchant is eligible only when it has positive sales in each of the two most recent three-month blocks.

For a selected portfolio represented by binary indicators $z_i$, expected monthly portfolio revenue is:

$$E_P = \sum_i z_i E_i$$

The revenue-only baseline is the 100 eligible merchants with the largest $E_i$. Each optimized portfolio must retain at least a chosen fraction $r$ of that baseline revenue:

$$E_P \ge r E_{baseline}$$

The `retention` field is this minimum constraint, while `portfolio_expected_fee_retention` is the amount actually achieved. Within each selected portfolio, merchants are sorted from highest to lowest `merchant_expected_monthly_gross_fees`; ABN is the deterministic tie-breaker.

## 2. Stability (cash-flow variability)

Stability is measured from **26 complete Monday–Sunday weeks** of merchant gross-fee history. Let $x_t$ be the vector of merchant fees in week $t$. The model estimates a Ledoit–Wolf shrinkage covariance matrix:

$$\widehat{\Sigma}_{shrunk} = (1-\alpha)\widehat{\Sigma}_{sample} + \alpha\,\mathrm{diag}(\widehat{\Sigma}_{sample})$$

Shrinkage reduces unstable off-diagonal covariance estimates while retaining each merchant's variance. For selection vector $z$:

$$SD_P = \sqrt{z^T\widehat{\Sigma}_{shrunk}z}$$

$$CV_P = \frac{SD_P}{\sum_i z_i\bar{x}_i}$$

Here $\bar{x}_i$ is merchant $i$'s historical weekly mean. The coefficient of variation (CV) makes volatility comparable relative to portfolio size: a **lower CV means more stable weekly fee generation**. CV is a ratio and is not constrained to the interval $[0,1]$.

## 3. Customer concentration

Customer concentration measures whether expected portfolio fees depend heavily on a small number of customers. Historical customer–merchant gross fees are rescaled merchant by merchant so that each merchant's customer allocations sum to its expected monthly fees. This gives exposure $A_{ci}$ from customer $c$ to merchant $i$. Shared customers are combined across all selected merchants:

$$F_c(z) = \sum_i A_{ci}z_i$$

Customer shares and the Herfindahl–Hirschman Index (HHI) are then:

$$s_c(z) = \frac{F_c(z)}{\sum_c F_c(z)}, \qquad HHI_P = \sum_c s_c(z)^2$$

HHI lies between 0 and 1. **Lower HHI means customer exposure is more diversified**; higher HHI means more reliance on a smaller customer base. The reciprocal, $1/HHI$, can be interpreted as the effective number of equally weighted customers.

## 4. Combined risk score and portfolio selection

CV and HHI use different scales, so each is divided by its value for the revenue-only top-100 baseline. With variability weight $w$ and concentration weight $1-w$:

$$Score_P = 100\left[w\frac{CV_P}{CV_{baseline}} + (1-w)\frac{HHI_P}{HHI_{baseline}}\right]$$

The delivered model uses $w=0.5$, so stability and customer concentration receive equal weight. The revenue-only baseline has a score of 100. A score below 100 indicates lower combined in-sample risk than that baseline. The score is **not a probability**.

For each retention level, the optimizer chooses exactly 100 merchants, respects the expected-revenue floor, and minimizes the combined score. `portfolio_score_lower_bound`, `portfolio_optimality_gap`, and `portfolio_certified` describe optimization evidence; certification concerns the numerical solution tolerance, not future performance.

## 5. Revenue and customer-concentration correlation matrix

The dashboard repeats portfolio metrics on each merchant row. The next cell first keeps one row per portfolio and then calculates the **Pearson correlation across the 30 delivered portfolios** (six dashboard scopes × five retention levels). This avoids incorrectly weighting a portfolio 100 times. Correlation is descriptive only and does not imply causation.

In [2]:
portfolio_rows = pd.concat(
    [
        frame.drop_duplicates("portfolio_id").assign(dashboard=label)
        for label, frame in dashboards.items()
    ],
    ignore_index=True,
)

correlation_data = portfolio_rows[[
    "portfolio_expected_monthly_gross_fees",
    "portfolio_customer_hhi",
]].rename(
    columns={
        "portfolio_expected_monthly_gross_fees": "Expected monthly revenue",
        "portfolio_customer_hhi": "Customer concentration (HHI)",
    }
)

correlation_matrix = correlation_data.corr(method="pearson")
display(correlation_matrix.round(3))

,Expected monthly revenue,Customer concentration (HHI)
Expected monthly revenue,1.000000,-0.588000
Customer concentration (HHI),-0.588000,1.000000


## 6. Top-100 rankings: first five merchants

Each dashboard contains five different top-100 portfolios—one per retention setting. The display below defaults to the 100% retention portfolio. Change `SELECTED_RETENTION` to `0.995`, `0.99`, `0.98`, or `0.95` to inspect another portfolio. `head(5)` shows ranks 1–5 after an explicit rank sort.

In [3]:
SELECTED_RETENTION = 1.0

RANKING_COLUMNS = [
    "rank_by_expected_fees",
    "merchant_abn",
    "merchant_name",
    "merchant_category",
    "merchant_expected_monthly_gross_fees",
]

def ranking_head(frame, retention=SELECTED_RETENTION):
    selected = frame.loc[np.isclose(frame["retention"], retention)].copy()
    if len(selected) != 100:
        raise ValueError(f"Expected 100 merchants at retention={retention}, found {len(selected)}")
    return selected.sort_values("rank_by_expected_fees")[RANKING_COLUMNS].head(5)

print(f"Overall portfolio — retention {SELECTED_RETENTION:.1%}")
display(ranking_head(dashboards["Overall"]))

Overall portfolio — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shop...","33,161.999483"
1,2,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses","31,088.087839"
2,3,32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops","31,053.347465"
3,4,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops","30,586.118562"
4,5,38700038932,Etiam Bibendum Industries,tent and awning shops,"30,106.623490"


In [4]:
for label, frame in dashboards.items():
    if label == "Overall":
        continue
    print(f"{label} — retention {SELECTED_RETENTION:.1%}")
    display(ranking_head(frame))

Arts, Crafts, Gifts & Entertainment — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops","31,053.347465"
1,2,45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops","29,233.139008"
2,3,63123845164,Odio Phasellus Institute,artist supply and craft shops,"28,213.797077"
3,4,64403598239,Lobortis Ultrices Company,"music shops - musical instruments, pianos, and...","27,879.515277"
4,5,94493496784,Dictum Phasellus In Institute,"gift, card, novelty, and souvenir shops","25,653.901521"


Automotive, Outdoor & Recreational Equipment — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,38700038932,Etiam Bibendum Industries,tent and awning shops,"30,106.623490"
1,2,96680767841,Ornare Limited,motor vehicle supplies and new parts,"28,873.185259"
2,3,89726005175,Est Nunc Consulting,tent and awning shops,"26,644.910948"
3,4,80551528183,Ac Ipsum LLC,tent and awning shops,"21,254.331719"
4,5,49891706470,Non Vestibulum Industries,tent and awning shops,"20,707.173292"


Fashion, Jewellery & Personal Care — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses","31,088.087839"
1,2,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops","30,586.118562"
2,3,49322182190,Gravida Mauris Incorporated,"watch, clock, and jewelry repair shops","24,839.190191"
3,4,19492220327,Commodo Ipsum Industries,"jewelry, watch, clock, and silverware shops","20,153.580080"
4,5,93558142492,Dolor Quisque Inc.,shoe shops,"15,058.810146"


Home, Garden & General Supplies — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shop...","33,161.999483"
1,2,76767266140,Phasellus At Limited,"furniture, home furnishings and equipment shop...","21,643.250122"
2,3,43186523025,Lorem Ipsum Sodales Industries,"florists supplies, nursery stock, and flowers","20,142.046513"
3,4,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers","18,225.235162"
4,5,57757792876,Pretium Et LLC,"stationery, office supplies and printing and w...","17,480.611380"


Technology, Digital & Communications — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,21439773999,Mauris Non Institute,"cable, satellite, and other pay television and...","28,634.957886"
1,2,82368304209,Nec Incorporated,telecom,"26,846.695224"
2,3,35909341340,Arcu Sed Eu Incorporated,"computer programming, data processing, and int...","22,760.744646"
3,4,72472909171,Nullam Consulting,"digital goods: books, movies, music","22,564.011261"
4,5,45433476494,Adipiscing Elit Foundation,"computers, computer peripheral equipment, and ...","20,784.249927"


## Interpretation notes

- Revenue is an expectation based on historical averages, not a guaranteed forecast.
- CV and HHI describe different risks: weekly cash-flow instability and dependence on shared customers.
- Relaxing the revenue-retention floor gives the optimizer more freedom to reduce risk, but actual achieved revenue retention may be above the minimum.
- Portfolio-level fields are repeated on all 100 merchant rows. Use one row per `portfolio_id` for portfolio analysis; only `merchant_expected_monthly_gross_fees` should be summed within a portfolio.
- The rankings order merchants inside an already selected portfolio by expected fees. They are not individual merchant risk rankings.

## 7. Presentation analysis: definitions and source checks

The following sections extend the original summary without rerunning optimisation. **Expected monthly gross return** means the final export's take-rate-adjusted monthly return **before consumer-fraud adjustment** (`merchant_expected_monthly_gross_fees`). The same definition is used for merchant and portfolio comparisons. It is not net profit. Curated `expected_revenue` is a different, fraud-adjusted historical total and is deliberately not substituted.

Here **α denotes the return-retention constraint**, not the covariance shrinkage parameter used earlier. The reference comparison is α = 100% versus 99%; other tables discover available alpha values from the files. A single snapshot, risk window, portfolio size and weight configuration is required so unrelated runs cannot be accidentally pooled.

Additional sources: `handoff/input_snapshot` for eligibility, customer exposures and weekly histories; `tables/curated/merchants` for merchant attributes; and `tables/curated/category_segments.parquet` for the official five-segment mapping (falling back to category membership in the per-segment final exports when that parquet is absent). Updating these inputs and rerunning the notebook recalculates all results. Inputs must belong to the same model run; a changed return definition requires corresponding model exports.

In [5]:
from IPython.display import Markdown

BASELINE_ALPHA = 1.0
COMPARISON_ALPHA = 0.99
SEGMENT_ORDER = [
    "Technology, Digital & Communications",
    "Home, Garden & General Supplies",
    "Fashion, Jewellery & Personal Care",
    "Arts, Crafts, Gifts & Entertainment",
    "Automotive, Outdoor & Recreational Equipment",
]
SNAPSHOT_DIR = PROJECT_ROOT / "handoff" / "input_snapshot"
GLOBAL = dashboards["Overall"].copy()
all_rankings = pd.concat([f.assign(scope=s) for s, f in dashboards.items()], ignore_index=True)
for column in ["as_of", "window_start", "window_end", "portfolio_size", "variability_weight", "concentration_weight"]:
    if all_rankings[column].nunique(dropna=False) != 1:
        raise ValueError(f"Select a single configuration before summarising: {column}")
for scope, frame in all_rankings.groupby("scope"):
    if frame.duplicated(["portfolio_id", "merchant_abn"]).any():
        raise ValueError(f"Duplicate merchant in portfolio: {scope}")
    if frame.groupby("retention")["portfolio_id"].nunique().ne(1).any():
        raise ValueError(f"Multiple portfolios at the same alpha: {scope}")
    for ident, rows in frame.groupby("portfolio_id"):
        if len(rows) != int(rows.portfolio_size.iloc[0]):
            raise ValueError(f"Incorrect portfolio size: {scope}, {ident}")
        if any(rows[c].nunique(dropna=False) != 1 for c in rows if c.startswith("portfolio_")):
            raise ValueError(f"Inconsistent portfolio metrics: {ident}")
        if not np.isclose(rows.merchant_expected_monthly_gross_fees.sum(), rows.portfolio_expected_monthly_gross_fees.iloc[0]):
            raise ValueError(f"Merchant and portfolio return totals disagree: {ident}")

metrics = GLOBAL.drop_duplicates("portfolio_id").set_index("retention").sort_index(ascending=False)
def alpha_key(value):
    matches = metrics.index[np.isclose(metrics.index, value)]
    if len(matches) != 1:
        raise ValueError(f"Required comparison alpha {value:g} is unavailable; update COMPARISON_ALPHA/BASELINE_ALPHA")
    return matches[0]
base_alpha, compare_alpha = alpha_key(BASELINE_ALPHA), alpha_key(COMPARISON_ALPHA)
base_metrics, compare_metrics = metrics.loc[base_alpha], metrics.loc[compare_alpha]

segment_path = PROJECT_ROOT / "tables/curated/category_segments.parquet"
if segment_path.exists():
    segment_map = pd.read_parquet(segment_path).explode("categories")
else:
    # The handoff may omit the mapping parquet. Per-segment exports provide
    # an authoritative fallback for categories represented in the rankings.
    segment_map = pd.concat([
        pd.DataFrame({"categories": frame.merchant_category.unique(), "segments": scope})
        for scope, frame in dashboards.items() if scope != "Overall"
    ], ignore_index=True)
    print("Segment mapping derived from per-segment final exports (mapping parquet absent).")
if segment_map.categories.duplicated().any() or set(segment_map.segments) != set(SEGMENT_ORDER):
    raise ValueError("Invalid category-to-segment mapping")
category_to_segment = segment_map.set_index("categories").segments
all_rankings["segment"] = all_rankings.merchant_category.map(category_to_segment)
if all_rankings.segment.isna().any():
    raise ValueError("Unmapped selected merchant categories")
for scope, rows in all_rankings.groupby("scope"):
    if scope != "Overall" and not rows.segment.eq(scope).all():
        raise ValueError(f"Merchant outside its declared segment: {scope}")

Segment mapping derived from per-segment final exports (mapping parquet absent).


## 8. Global return–risk trade-offs

Return sacrificed is the baseline monthly gross return minus the selected portfolio's monthly gross return. The combined risk score is an index with baseline 100, not a loss probability or monetary saving. Solver certification and gaps are retained because “best found” portfolios are not necessarily certified optimal.

In [6]:
tradeoffs = pd.DataFrame({
    "Expected monthly gross return (AUD)": metrics.portfolio_expected_monthly_gross_fees,
    "Monthly return sacrificed vs baseline (AUD)": base_metrics.portfolio_expected_monthly_gross_fees - metrics.portfolio_expected_monthly_gross_fees,
    "Combined risk index": metrics.portfolio_risk_score,
    "Certified": metrics.portfolio_certified,
    "Gap upper bound (%)": metrics.portfolio_optimality_gap * 100,
})
tradeoffs.index = [f"α = {a*100:g}%" for a in tradeoffs.index]
display(tradeoffs.style.format({
    "Expected monthly gross return (AUD)": "A${:,.0f}",
    "Monthly return sacrificed vs baseline (AUD)": "A${:,.0f}",
    "Combined risk index": "{:.2f}", "Gap upper bound (%)": "{:.3f}",
}))

,Expected monthly gross return (AUD),Monthly return sacrificed vs baseline (AUD),Combined risk index,Certified,Gap upper bound (%)
α = 100%,"A$1,666,400",A$0,100.00,True,0.000
α = 99.5%,"A$1,658,375","A$8,024",99.32,True,0.076
α = 99%,"A$1,649,905","A$16,494",98.62,False,0.115
α = 98%,"A$1,633,626","A$32,774",97.66,False,0.159
α = 95%,"A$1,583,307","A$83,093",95.87,False,0.912


## 9. Merchant substitutions and change in portfolio metrics

“Removed” and “added” describe the change from the baseline to the comparison alpha. Changes in portfolio risk cannot be attributed to individual merchants independently: covariance and shared customers are evaluated jointly. Risk-index points and relative percentage reductions are reported separately.

In [7]:
base_rows = GLOBAL.loc[np.isclose(GLOBAL.retention, base_alpha)].set_index("merchant_abn")
comparison_rows = GLOBAL.loc[np.isclose(GLOBAL.retention, compare_alpha)].set_index("merchant_abn")
removed_ids = set(base_rows.index) - set(comparison_rows.index)
added_ids = set(comparison_rows.index) - set(base_rows.index)
changes = pd.concat([
    base_rows.loc[sorted(removed_ids)].assign(Change="Removed"),
    comparison_rows.loc[sorted(added_ids)].assign(Change="Added"),
]).reset_index()
changes = changes[["Change", "merchant_abn", "merchant_name", "merchant_category", "merchant_expected_monthly_gross_fees"]].rename(columns={"merchant_expected_monthly_gross_fees": "Expected monthly gross return (AUD)"})
display(changes.style.format({"Expected monthly gross return (AUD)": "A${:,.2f}"}))
comparison_specs = {
    "Monthly gross return (AUD)": "portfolio_expected_monthly_gross_fees",
    "Combined risk index": "portfolio_risk_score",
    "Weekly return CV": "portfolio_variability_cv",
    "Consumer concentration (HHI)": "portfolio_customer_hhi",
}
comparison = pd.DataFrame([
    {"Metric": label, "Baseline": float(base_metrics[key]), "Comparison": float(compare_metrics[key]),
     "Change (comparison − baseline)": float(compare_metrics[key] - base_metrics[key]),
     "Relative reduction (%)": (1-float(compare_metrics[key])/float(base_metrics[key]))*100 if base_metrics[key] else np.nan}
    for label, key in comparison_specs.items()
]).set_index("Metric")
display(comparison)
return_lost = float(base_metrics.portfolio_expected_monthly_gross_fees - compare_metrics.portfolio_expected_monthly_gross_fees)
risk_reduction = float(base_metrics.portfolio_risk_score - compare_metrics.portfolio_risk_score)
display(Markdown(f"At **α = {compare_alpha:.3%}** versus **{base_alpha:.3%}**, **{len(removed_ids)}** merchants are removed and **{len(added_ids)}** added. Monthly gross return changes by **A${-return_lost:,.2f}**; the combined risk index changes by **{-risk_reduction:.3f} points**. These are in-sample trade-offs, not demonstrated reductions in realised losses."))
assert np.isclose(base_rows.loc[sorted(removed_ids), "merchant_expected_monthly_gross_fees"].sum() - comparison_rows.loc[sorted(added_ids), "merchant_expected_monthly_gross_fees"].sum(), return_lost)

,Change,merchant_abn,merchant_name,merchant_category,Expected monthly gross return (AUD)
0,Removed,19492220327,Commodo Ipsum Industries,"jewelry, watch, clock, and silverware shops","A$20,153.58"
1,Removed,35809331583,Euismod In LLC,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","A$9,604.20"
2,Added,53126107676,Sed Neque Corporation,"florists supplies, nursery stock, and flowers","A$5,642.98"
3,Added,61080648284,Lobortis Ultrices LLC,artist supply and craft shops,"A$7,620.39"


,Baseline,Comparison,Change (comparison − baseline),Relative reduction (%)
Metric,,,,
Monthly gross return (AUD),"1,666,399.534895","1,649,905.124237","-16,494.410658",0.989823
Combined risk index,100.000000,98.620704,-1.379296,1.379296
Weekly return CV,0.031747,0.031418,-0.000329,1.035696
Consumer concentration (HHI),0.000047,0.000046,-0.000001,1.722896


At **α = 99.000%** versus **100.000%**, **2** merchants are removed and **2** added. Monthly gross return changes by **A$-16,494.41**; the combined risk index changes by **-1.379 points**. These are in-sample trade-offs, not demonstrated reductions in realised losses.

## 10. Segment composition of every ranking

Percentages represent **merchant counts**, not return weights. All five segments are displayed, including zero counts. Global counts equal percentages when portfolio size is 100; both are calculated independently so this remains valid if portfolio sizes change. Each per-segment portfolio should contain 100% merchants from its own segment.

In [8]:
segment_counts_by_scope = {}
segment_percentages_by_scope = {}
for scope in dashboards:
    rows = all_rankings.loc[all_rankings.scope.eq(scope)]
    counts = pd.crosstab(rows.segment, rows.retention).reindex(index=SEGMENT_ORDER, columns=sorted(rows.retention.unique()), fill_value=0)
    percentages = counts.div(counts.sum(axis=0), axis=1) * 100
    assert np.allclose(percentages.sum(axis=0), 100)
    segment_counts_by_scope[scope] = counts
    segment_percentages_by_scope[scope] = percentages
    display(Markdown(f"**{scope}: merchant proportions**"))
    display(percentages.rename(columns=lambda a: f"α = {a*100:g}%").style.format("{:.1f}%"))
display(Markdown("**Global merchant counts (presentation table)**"))
display(segment_counts_by_scope["Overall"].rename(columns=lambda a: f"α = {a*100:g}%"))

**Overall: merchant proportions**

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",17.0%,18.0%,17.0%,17.0%,17.0%
"Home, Garden & General Supplies",17.0%,17.0%,19.0%,18.0%,19.0%
"Fashion, Jewellery & Personal Care",14.0%,14.0%,14.0%,15.0%,15.0%
"Arts, Crafts, Gifts & Entertainment",31.0%,30.0%,28.0%,28.0%,27.0%
"Automotive, Outdoor & Recreational Equipment",21.0%,21.0%,22.0%,22.0%,22.0%


**Arts, Crafts, Gifts & Entertainment: merchant proportions**

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",0.0%,0.0%,0.0%,0.0%,0.0%
"Home, Garden & General Supplies",0.0%,0.0%,0.0%,0.0%,0.0%
"Fashion, Jewellery & Personal Care",0.0%,0.0%,0.0%,0.0%,0.0%
"Arts, Crafts, Gifts & Entertainment",100.0%,100.0%,100.0%,100.0%,100.0%
"Automotive, Outdoor & Recreational Equipment",0.0%,0.0%,0.0%,0.0%,0.0%


**Automotive, Outdoor & Recreational Equipment: merchant proportions**

...

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",0.0%,0.0%,0.0%,0.0%,0.0%
"Home, Garden & General Supplies",100.0%,100.0%,100.0%,100.0%,100.0%
"Fashion, Jewellery & Personal Care",0.0%,0.0%,0.0%,0.0%,0.0%
"Arts, Crafts, Gifts & Entertainment",0.0%,0.0%,0.0%,0.0%,0.0%
"Automotive, Outdoor & Recreational Equipment",0.0%,0.0%,0.0%,0.0%,0.0%


**Technology, Digital & Communications: merchant proportions**

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",100.0%,100.0%,100.0%,100.0%,100.0%
"Home, Garden & General Supplies",0.0%,0.0%,0.0%,0.0%,0.0%
"Fashion, Jewellery & Personal Care",0.0%,0.0%,0.0%,0.0%,0.0%
"Arts, Crafts, Gifts & Entertainment",0.0%,0.0%,0.0%,0.0%,0.0%
"Automotive, Outdoor & Recreational Equipment",0.0%,0.0%,0.0%,0.0%,0.0%


**Global merchant counts (presentation table)**

retention,α = 95%,α = 98%,α = 99%,α = 99.5%,α = 100%
segment,,,,,
"Technology, Digital & Communications",17,18,17,17,17
"Home, Garden & General Supplies",17,17,19,18,19
"Fashion, Jewellery & Personal Care",14,14,14,15,15
"Arts, Crafts, Gifts & Entertainment",31,30,28,28,27
"Automotive, Outdoor & Recreational Equipment",21,21,22,22,22


## 11. Core merchants and their contribution to return

Always selected = present at every available global alpha. Sometimes selected = present at least once but not at every alpha. Never selected = eligible in the frozen input snapshot but absent from all global portfolios. Ineligible merchants are excluded from this comparison. These are selection-frequency groups, not automatic acceptance/rejection labels.

In [9]:
snapshot_merchants = pd.read_parquet(SNAPSHOT_DIR / "merchants.parquet")
snapshot_merchants["merchant_abn"] = snapshot_merchants.merchant_abn.astype("string")
if snapshot_merchants.merchant_abn.duplicated().any():
    raise ValueError("Duplicate snapshot merchant ABN")
if not snapshot_merchants.eligible.isin([True, False]).all():
    raise ValueError("Snapshot eligibility must be boolean")
eligible = snapshot_merchants.loc[snapshot_merchants.eligible.eq(True)].set_index("merchant_abn").copy()
if not set(GLOBAL.merchant_abn).issubset(eligible.index):
    raise ValueError("Final ranking includes merchant absent from eligible snapshot")
snapshot_returns = GLOBAL.merchant_abn.map(eligible.expected_monthly_fees)
if not np.allclose(snapshot_returns, GLOBAL.merchant_expected_monthly_gross_fees):
    raise ValueError("Final returns disagree with frozen snapshot")
selection_frequency = GLOBAL.groupby("merchant_abn").retention.nunique()
n_alphas = GLOBAL.retention.nunique()
eligible["selection_count"] = selection_frequency.reindex(eligible.index, fill_value=0)
eligible["Selection group"] = np.select([eligible.selection_count.eq(n_alphas), eligible.selection_count.gt(0)], ["Always selected", "Sometimes selected"], default="Never selected")
core = eligible.loc[eligible["Selection group"].eq("Always selected")].copy()
selected_union = int(eligible.selection_count.gt(0).sum())
core_return = comparison_rows.loc[comparison_rows.index.isin(core.index), "merchant_expected_monthly_gross_fees"].sum()
core_return_share = core_return / compare_metrics.portfolio_expected_monthly_gross_fees
alpha_memberships = GLOBAL.groupby("merchant_abn").retention.apply(lambda v: ", ".join(f"{a*100:g}%" for a in sorted(v.unique(), reverse=True)))
core["Selected at α"] = alpha_memberships.reindex(core.index)
display(Markdown(f"**{selected_union:,}** of **{len(eligible):,}** eligible merchants occur in at least one global portfolio; **{len(core):,}** occur in all **{n_alphas}**. The core contributes **A${core_return:,.2f}**, or **{core_return_share:.2%}**, of monthly gross return at **α = {compare_alpha*100:g}%**."))
display(core.sort_values("expected_monthly_fees", ascending=False)[["name", "category", "expected_monthly_fees", "Selected at α"]].rename(columns={"expected_monthly_fees": "Expected monthly gross return (AUD)"}).style.format({"Expected monthly gross return (AUD)": "A${:,.2f}"}))
core_segments = core.category.map(category_to_segment).value_counts().reindex(SEGMENT_ORDER, fill_value=0)
display(core_segments.rename("Core merchant count").to_frame())

**109** of **3,897** eligible merchants occur in at least one global portfolio; **91** occur in all **5**. The core contributes **A$1,533,599.23**, or **92.95%**, of monthly gross return at **α = 99%**.

,name,category,Expected monthly gross return (AUD),Selected at α
merchant_abn,,,,
79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shops, and manufacturers, except appliances","A$33,162.00","100%, 99.5%, 99%, 98%, 95%"
48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses","A$31,088.09","100%, 99.5%, 99%, 98%, 95%"
32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops","A$31,053.35","100%, 99.5%, 99%, 98%, 95%"
86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops","A$30,586.12","100%, 99.5%, 99%, 98%, 95%"
38700038932,Etiam Bibendum Industries,tent and awning shops,"A$30,106.62","100%, 99.5%, 99%, 98%, 95%"
45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops","A$29,233.14","100%, 99.5%, 99%, 98%, 95%"
96680767841,Ornare Limited,motor vehicle supplies and new parts,"A$28,873.19","100%, 99.5%, 99%, 98%, 95%"
21439773999,Mauris Non Institute,"cable, satellite, and other pay television and radio services","A$28,634.96","100%, 99.5%, 99%, 98%, 95%"
63123845164,Odio Phasellus Institute,artist supply and craft shops,"A$28,213.80","100%, 99.5%, 99%, 98%, 95%"


,Core merchant count
category,
"Technology, Digital & Communications",15
"Home, Garden & General Supplies",16
"Fashion, Jewellery & Personal Care",14
"Arts, Crafts, Gifts & Entertainment",25
"Automotive, Outdoor & Recreational Equipment",21


## 12. What distinguishes always-, sometimes- and never-selected merchants?

All statistics below are **merchant-level medians**, not pooled customer statistics. Customer counts and largest-customer shares use customer–merchant return over the snapshot's risk window. Weekly variability is individual sample standard deviation (`ddof=1`) divided by mean weekly return, including zero weeks; it is **not** the shrunk portfolio CV. Largest consumer's share is maximum customer return divided by the merchant's total customer return, distinct from portfolio HHI.

Take rates and average fraud scores are stored in percentage points. Curated average transaction values and fraud scores cover the stored history, which may extend beyond the frozen model snapshot; they are descriptive attributes, not evidence of out-of-sample prediction. Fraud scores are not realised fraud rates. No fraud adjustment is applied to the return figures in this table.

In [10]:
weekly = pd.read_parquet(SNAPSHOT_DIR / "weekly_fees.parquet")
customers = pd.read_parquet(SNAPSHOT_DIR / "customer_fees.parquet")
for frame in [weekly, customers]:
    frame["merchant_abn"] = frame.merchant_abn.astype("string")
if weekly.duplicated(["merchant_abn", "week_end"]).any() or customers.duplicated(["merchant_abn", "consumer_id"]).any():
    raise ValueError("Duplicate weekly or customer–merchant rows")
weekly["week_end"] = pd.to_datetime(weekly.week_end)
start, end = pd.Timestamp(base_metrics.window_start), pd.Timestamp(base_metrics.window_end)
if not weekly.week_end.between(start, end).all():
    raise ValueError("Weekly histories extend outside portfolio risk window")
week_count = len(pd.date_range(start, end, freq="W-SUN"))
if not weekly.groupby("merchant_abn").size().reindex(eligible.index, fill_value=0).eq(week_count).all():
    raise ValueError("Missing weekly rows: genuine zero weeks must be present")
weekly_stats = weekly.groupby("merchant_abn").gross_fees.agg(["mean", "std"])
weekly_stats["weekly_cv"] = weekly_stats["std"].div(weekly_stats["mean"].where(weekly_stats["mean"].gt(0)))
customer_stats = customers.groupby("merchant_abn").agg(customers=("consumer_id", "nunique"), total=("gross_fees", "sum"), largest=("gross_fees", "max"))
customer_stats["largest_share"] = customer_stats.largest.div(customer_stats.total.where(customer_stats.total.gt(0)))
curated = pd.read_parquet(PROJECT_ROOT / "tables/curated/merchants", columns=["merchant_abn", "take_rate", "avg_value_per_transaction", "avg_consumer_fraud_prob"])
curated["merchant_abn"] = curated.merchant_abn.astype("string")
if curated.merchant_abn.duplicated().any() or not set(eligible.index).issubset(curated.merchant_abn):
    raise ValueError("Curated merchant join is incomplete or duplicated")
profiles = eligible[["name", "Selection group", "selection_count", "expected_monthly_fees"]].join(weekly_stats[["weekly_cv"]]).join(customer_stats[["customers", "largest_share"]]).join(curated.set_index("merchant_abn"))
profiles["customers"] = profiles.customers.fillna(0)
features = {
    "Expected monthly gross return": ("expected_monthly_fees", "A${:,.0f}"),
    f"Customers over {week_count} weeks": ("customers", "{:,.0f}"),
    "Weekly return variability": ("weekly_cv", "{:.1%}"),
    "Largest consumer’s share": ("largest_share", "{:.2%}"),
    "Merchant take rate": ("take_rate", "{:.2f}%"),
    "Average transaction value": ("avg_value_per_transaction", "A${:,.0f}"),
    "Average consumer fraud score": ("avg_consumer_fraud_prob", "{:.2f}%"),
}
group_order = ["Always selected", "Sometimes selected", "Never selected"]
group_counts = profiles["Selection group"].value_counts().reindex(group_order, fill_value=0)
summary_medians = pd.DataFrame({g: {label: profiles.loc[profiles["Selection group"].eq(g), col].median() for label, (col, fmt) in features.items()} for g in group_order})
# Presentation uses conventional half-up rounding for whole-customer medians.
from decimal import Decimal, ROUND_HALF_UP
def format_median(label, value, fmt):
    if pd.isna(value):return "—"
    if label.startswith("Customers over"):
        return f"{Decimal(str(value)).quantize(Decimal('1'), rounding=ROUND_HALF_UP):,}"
    return fmt.format(value)
summary_display = pd.DataFrame({f"{g} ({group_counts[g]:,})": {label: format_median(label, summary_medians.loc[label, g], fmt) for label, (col, fmt) in features.items()} for g in group_order})
display(summary_display.rename_axis("Summary statistics (median)"))
missing_counts = profiles.groupby("Selection group")[[col for col, fmt in features.values()]].apply(lambda f: f.isna().sum()).reindex(group_order, fill_value=0)
if missing_counts.to_numpy().any():
    display(Markdown("**Missing attributes excluded from each median:**"))
    display(missing_counts)
assert group_counts.sum() == len(eligible)
assert group_counts["Always selected"] + group_counts["Sometimes selected"] == selected_union

display(Markdown("**Sometimes-selected merchants and their exact alpha memberships**"))
sometimes = profiles.loc[profiles["Selection group"].eq("Sometimes selected"), ["name", "selection_count"]].copy()
sometimes["Selected at α"] = alpha_memberships.reindex(sometimes.index)
display(sometimes.sort_values(["selection_count", "name"], ascending=[False, True]))

,Always selected (91),Sometimes selected (18),"Never selected (3,788)"
Summary statistics (median),,,
Expected monthly gross return,"A$15,059","A$8,087",A$258
Customers over 26 weeks,"6,486","1,353",136
Weekly return variability,9.6%,17.9%,54.3%
Largest consumer’s share,0.10%,0.37%,2.77%
Merchant take rate,5.53%,5.69%,4.48%
Average transaction value,A$302,A$643,A$301
Average consumer fraud score,4.89%,5.72%,4.88%


**Sometimes-selected merchants and their exact alpha memberships**

,name,selection_count,Selected at α
merchant_abn,,,
46012371285,A Ultricies Inc.,4,"100%, 99.5%, 99%, 98%"
31334588839,Lacus Aliquam Corporation,4,"100%, 99.5%, 99%, 98%"
61080648284,Lobortis Ultrices LLC,4,"99.5%, 99%, 98%, 95%"
38212167834,Magna Suspendisse Industries,4,"100%, 99.5%, 99%, 98%"
82368304209,Nec Incorporated,4,"100%, 99.5%, 99%, 98%"
94455880010,Nunc Sit LLC,4,"100%, 99.5%, 99%, 98%"
12771097467,At Pretium Corp.,3,"100%, 99.5%, 99%"
53126107676,Sed Neque Corporation,3,"99.5%, 99%, 95%"
39649557865,Arcu Morbi Institute,2,"98%, 95%"
